# Task 2: Incremental CPG Parser Service

**Mục tiêu**: Xây dựng một Python service đọc từng file source code thay vì đọc cả repo cùng lúc (hoạt động với bounded memory), dùng thư viện để trích xuất các node và cạnh (AST, CFG, DFG, CALL) cho Code Property Graph, sau đó xuất ra định dạng event với định danh ổn định (stable identifier).


## Cách tiếp cận & Công nghệ

- **Thư viện parse**: Sử dụng module `ast` có sẵn của Python. Lý do: Dễ cài đặt, không cần dependency mở rộng như `tree-sitter` hay server riêng như `Joern`, hoàn toàn đủ dùng cho mục đích xây dựng CPG cơ bản của bài lab.
- **Bounded Memory**: Hệ thống đọc và parse từng file độc lập (hàm `parse_file` trong `cpg_visitor.py`). Khi parse xong một file, service sẽ giải phóng bộ nhớ của cây AST, đảm bảo RAM không bị đầy kể cả khi parse repo hàng nghìn file.
- **Stable Identifiers**: Thay vì dùng UUID ngẫu nhiên hay ID mặc định của object trong bộ nhớ, `node_id` và `edge_id` được sinh ra qua cơ chế hash (hàm `make_node_id`, `make_edge_id` trong `stable_id.py`). Đầu vào của hash bao gồm `file_path`, scope và type. Cách này đảm bảo tính lũy đẳng (idempotent), hỗ trợ reprocessing ở Task 6.

### Sơ đồ Kiến trúc Incremental CPG Parser Service

![Task 2 Incremental CPG Parser Architecture](images/02_cpg_parser_architecture.png)



## Minh hoạ kết quả Parse trên 1 file
Để chứng minh việc trích xuất AST/CFG/DFG hoạt động tốt, chúng ta sẽ thử import hàm `parse_source` và parse một đoạn code mẫu trực tiếp.

In [1]:
import sys
from pathlib import Path

# Thêm đường dẫn vào thư mục parser-service để import
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / "parser-service"))

from cpg_visitor import parse_source, SAMPLE

# Chạy thử hàm parse_source với đoạn code SAMPLE có sẵn trong cpg_visitor.py
nodes, edges, meta = parse_source(SAMPLE, file_path="<sample>", repo_commit="dev")

print(f"Tổng số nodes trích xuất: {len(nodes)}")
print(f"Tổng số edges trích xuất: {len(edges)}")


Tổng số nodes trích xuất: 10
Tổng số edges trích xuất: 19


## Xem thử mẫu Data Event
Kiểm tra xem dữ liệu có đầy đủ cấu trúc và ID cố định không.

In [2]:
import json

print("--- SAMPLE NODE EVENT ---")
print(json.dumps(nodes[0], indent=2, ensure_ascii=False))

print("\n--- SAMPLE EDGE EVENT ---")
print(json.dumps(edges[0], indent=2, ensure_ascii=False))


--- SAMPLE NODE EVENT ---
{
  "schema_version": "v1",
  "event_timestamp": "2026-07-24T10:56:44.837603+00:00",
  "node_id": "52504a68222fce273fcf4661",
  "node_type": "Module",
  "name": null,
  "file_path": "<sample>",
  "line_start": 0,
  "line_end": 0,
  "col_start": null,
  "col_end": null,
  "repo_commit": "dev"
}

--- SAMPLE EDGE EVENT ---
{
  "schema_version": "v1",
  "event_timestamp": "2026-07-24T10:56:44.837603+00:00",
  "edge_id": "b0f33f26e186a2c77b844115",
  "edge_type": "AST",
  "source_node_id": "52504a68222fce273fcf4661",
  "target_node_id": "76e4e9e8777ae41940544197",
  "dfg_variable": null,
  "file_path": "<sample>",
  "repo_commit": "dev"
}


## Reflection

- **What worked**: Sử dụng `ast` module kết hợp với mô hình `NodeVisitor` giúp việc duyệt qua code Python rất mượt và nhanh gọn. Thiết kế Stable Identifier (sinh ID tĩnh theo hash path+thứ tự) hoạt động chuẩn xác.
- **What failed / Challenges**: Việc tự xây dựng thuật toán nối cạnh Control Flow Graph (CFG) và Data Flow Graph (DFG) khá nhằn vì cấu trúc lồng nhau của AST Python rất phức tạp.
- **Resolution**: Nhóm quyết định giới hạn việc xây dựng CFG và DFG ở các cấu trúc cơ bản nhất (`If`, `For`, `While`, `Assign`, `Call`) để phục vụ đúng tính chất PoC thay vì tạo ra công cụ phân tích tĩnh toàn diện. Điều này giúp tối ưu tiến độ và vẫn đảm bảo dữ liệu đồ thị có ý nghĩa.